# Advanced GPU Project 1 — Fine-tune a Vision Transformer (ViT)
## Image classification with Hugging Face + PyTorch

**Task:** Classify bean leaf images using transfer learning from a pretrained Vision Transformer.  
**Dataset:** Hugging Face `AI-Lab-Makerere/beans`  
**Runtime:** Google Colab GPU recommended (T4 or better). Training can run on CPU but will be much slower.

### Learning outcomes
- Load a real image dataset
- Fine-tune a pretrained image model
- Track validation metrics
- Evaluate on a held-out test set
- Inspect errors and save the trained model

> In Colab, choose **Runtime → Change runtime type → T4 GPU** if available.

## 1. Verify GPU

In [ ]:
import torch, platform
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU is not active. Enable a GPU runtime in Colab before training.")

In [ ]:
%pip -q install -U transformers datasets accelerate scikit-learn pillow torchvision

## 2. Load the Hugging Face dataset

In [ ]:
from datasets import load_dataset, DatasetDict
from collections import Counter

raw = load_dataset("AI-Lab-Makerere/beans")
print(raw)
display(raw["train"][0]["image"])
label_names = raw["train"].features["labels"].names
print("Labels:", label_names)

## 3. Create train, validation, and test splits

In [ ]:
split = raw["train"].train_test_split(
    test_size=0.15, seed=42, stratify_by_column="labels"
)
ds = DatasetDict({
    "train": split["train"],
    "validation": split["test"],
    "test": raw["test"]
})
print({k: len(v) for k, v in ds.items()})
print("Training label counts:", Counter(ds["train"]["labels"]))

## 4. Load a pretrained Vision Transformer

In [ ]:
from transformers import AutoImageProcessor, AutoModelForImageClassification

checkpoint = "google/vit-base-patch16-224-in21k"
processor = AutoImageProcessor.from_pretrained(checkpoint)
model = AutoModelForImageClassification.from_pretrained(
    checkpoint,
    num_labels=len(label_names),
    id2label={i: n for i, n in enumerate(label_names)},
    label2id={n: i for i, n in enumerate(label_names)},
    ignore_mismatched_sizes=True
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print("Model device:", device)

## 5. Preprocess images and build batches

In [ ]:
from torchvision.transforms import (
    Compose, RandomResizedCrop, RandomHorizontalFlip, Resize, CenterCrop,
    ToTensor, Normalize
)
from torch.utils.data import DataLoader

size = processor.size.get("height", 224) if isinstance(processor.size, dict) else 224
normalize = Normalize(mean=processor.image_mean, std=processor.image_std)
train_tf = Compose([
    RandomResizedCrop(size), RandomHorizontalFlip(), ToTensor(), normalize
])
eval_tf = Compose([Resize(size), CenterCrop(size), ToTensor(), normalize])

def train_transform(batch):
    batch["pixel_values"] = [train_tf(img.convert("RGB")) for img in batch["image"]]
    return batch

def eval_transform(batch):
    batch["pixel_values"] = [eval_tf(img.convert("RGB")) for img in batch["image"]]
    return batch

train_ds = ds["train"].with_transform(train_transform)
val_ds = ds["validation"].with_transform(eval_transform)
test_ds = ds["test"].with_transform(eval_transform)

def collate_fn(examples):
    return {
        "pixel_values": torch.stack([e["pixel_values"] for e in examples]),
        "labels": torch.tensor([e["labels"] for e in examples], dtype=torch.long)
    }

## 6. Fine-tune the model on the GPU

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from transformers import TrainingArguments, Trainer

def metrics(eval_pred):
    logits, labels = eval_pred
    pred = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, pred),
        "macro_f1": f1_score(labels, pred, average="macro")
    }

args = TrainingArguments(
    output_dir="vit-beans-checkpoints",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_steps=20,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=42
)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collate_fn,
    compute_metrics=metrics
)
trainer.train()

## 7. Evaluate once on the held-out test set

In [ ]:
test_result = trainer.predict(test_ds)
y_true = test_result.label_ids
y_pred = np.argmax(test_result.predictions, axis=-1)
print("Test metrics:", test_result.metrics)

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

print(classification_report(y_true, y_pred, target_names=label_names, zero_division=0))
ConfusionMatrixDisplay.from_predictions(
    y_true, y_pred, display_labels=label_names, xticks_rotation=45, cmap="Blues"
)
plt.tight_layout()
plt.show()

## 8. Predict an example image

In [ ]:
sample = ds["test"][0]
image = sample["image"].convert("RGB")
display(image)
inputs = processor(images=image, return_tensors="pt").to(device)
model.eval()
with torch.no_grad():
    logits = model(**inputs).logits
prob = torch.softmax(logits, dim=-1)[0]
idx = int(prob.argmax())
print("Predicted:", label_names[idx], f"({prob[idx].item():.1%})")
print("Actual:", label_names[sample["labels"]])

## 9. Save the fine-tuned model

In [ ]:
trainer.save_model("vit-beans-model")
processor.save_pretrained("vit-beans-model")
print("Saved model to vit-beans-model/")

## Challenge extensions
- Compare full fine-tuning with a frozen backbone.
- Try different learning rates and batch sizes.
- Inspect the most confident incorrect predictions.
- Add per-class F1 and an error analysis.
- Build a Gradio image upload demo.

**Submit:** notebook, test metrics, confusion matrix, model artifact or Hub link, training time, GPU type, and limitations.